In [ ]:
# ============================================================
# Baseline NN-only Fcr
# Parameter-matched with proposed PCNN: 135,171 params
# Normalization exactly like proposed model:
#   X   -> scX
#   Fcr -> scF
# Model predicts scaled Fcr
# ============================================================

import os
import json
import time
import random
import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# ============================================================
# Config
# ============================================================

CSV_PATH = "DatasetMTH_coeffs_DT123_5levels.csv"

OUT_DIR = "baseline_Fcr_only_param_matched"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)
os.makedirs(f"{OUT_DIR}/scalers", exist_ok=True)

VAR_COLS = [f"var{i}" for i in range(1, 11)]
FCR_COL = "Fcr"

# Exact parameter-matched single-output MLP:
# 10 -> 238 -> 263 -> 263 -> 1 = 135,171 params
ENC = [238, 263, 263]

ACT = "relu"
DROPOUT = 0.0
BATCHNORM = False

EPOCHS = 200
BATCH_SIZE = 4096
LR = 1e-3
WEIGHT_DECAY = 1e-4

TEST_SIZE = 0.2
VAL_SIZE = 0.1
SEED = 42

ES_PATIENCE = 8
ES_MIN_DELTA = 0.5e-5

LR_FACTOR = 0.5
LR_PATIENCE = 4
LR_MIN = 1e-6

USE_AMP = True
GRAD_CLIP = 1.0

device = "cuda" if torch.cuda.is_available() else "cpu"
pin_memory = device.startswith("cuda")


# ============================================================
# Utils
# ============================================================

def seed_everything(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True


def sync():
    if device.startswith("cuda"):
        torch.cuda.synchronize()


def get_act(name):
    name = name.lower()

    if name == "relu":
        return nn.ReLU(inplace=True)
    if name == "gelu":
        return nn.GELU()
    if name == "silu":
        return nn.SiLU(inplace=True)
    if name == "tanh":
        return nn.Tanh()
    if name == "leaky_relu":
        return nn.LeakyReLU(0.1, inplace=True)

    raise ValueError(f"Unknown activation: {name}")


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def check_finite_array(name, arr):
    arr = np.asarray(arr)
    if not np.isfinite(arr).all():
        n_nan = np.isnan(arr).sum()
        n_inf = np.isinf(arr).sum()
        raise ValueError(f"{name} contains NaN/Inf: NaN={n_nan}, Inf={n_inf}")


def make_loader(X, y, batch_size, shuffle):
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)

    ds = TensorDataset(X, y)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        pin_memory=pin_memory,
        num_workers=0,
    )


# ============================================================
# Model
# ============================================================

class FcrOnlyMLP(nn.Module):
    def __init__(self, in_dim, enc, act="gelu", dropout=0.0, batchnorm=False):
        super().__init__()

        layers = []
        prev = in_dim

        for h in enc:
            layers.append(nn.Linear(prev, h))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(get_act(act))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def eval_mse_scaled(model, loader):
    model.eval()

    sse = 0.0
    n = 0

    for Xb, yb in loader:
        Xb = Xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        pred = model(Xb)

        if not torch.isfinite(pred).all():
            raise FloatingPointError("Prediction contains NaN/Inf during evaluation.")

        loss_sum = torch.sum((pred - yb) ** 2)

        if not torch.isfinite(loss_sum):
            raise FloatingPointError("Eval loss contains NaN/Inf.")

        sse += loss_sum.item()
        n += Xb.size(0)

    return sse / max(n, 1)


@torch.no_grad()
def predict_original_scale(model, loader, scF):
    model.eval()

    y_true_s_all = []
    y_pred_s_all = []

    for Xb, yb in loader:
        Xb = Xb.to(device, non_blocking=True)
        pred_s = model(Xb).detach().cpu().numpy()

        y_pred_s_all.append(pred_s)
        y_true_s_all.append(yb.numpy())

    y_true_s = np.vstack(y_true_s_all)
    y_pred_s = np.vstack(y_pred_s_all)

    y_true = scF.inverse_transform(y_true_s).reshape(-1)
    y_pred = scF.inverse_transform(y_pred_s).reshape(-1)

    return y_true, y_pred


def compute_metrics(y_true, y_pred):
    abs_err = np.abs(y_pred - y_true)
    rel_err = abs_err / np.maximum(np.abs(y_true), 1e-12)

    return {
        "Fcr_MAE": float(mean_absolute_error(y_true, y_pred)),
        "Fcr_RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "Fcr_R2": float(r2_score(y_true, y_pred)),

        "Fcr_mean_abs_error": float(np.mean(abs_err)),
        "Fcr_median_abs_error": float(np.median(abs_err)),
        "Fcr_p95_abs_error": float(np.percentile(abs_err, 95)),
        "Fcr_max_abs_error": float(np.max(abs_err)),

        "Fcr_mean_rel_error": float(np.mean(rel_err)),
        "Fcr_median_rel_error": float(np.median(rel_err)),
        "Fcr_p95_rel_error": float(np.percentile(rel_err, 95)),
        "Fcr_max_rel_error": float(np.max(rel_err)),
    }


@torch.no_grad()
def forward_time_fullset(model, loader, n_warmup_batches=5):
    model.eval()

    # warm-up
    it = iter(loader)
    for _ in range(n_warmup_batches):
        try:
            Xb, _ = next(it)
        except StopIteration:
            break

        Xb = Xb.to(device, non_blocking=True)
        _ = model(Xb)

    sync()

    t0 = time.perf_counter()

    n = 0
    for Xb, _ in loader:
        Xb = Xb.to(device, non_blocking=True)
        _ = model(Xb)
        n += Xb.size(0)

    sync()

    t1 = time.perf_counter()

    return t1 - t0, n


# ============================================================
# Main
# ============================================================

seed_everything(SEED, deterministic=False)

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

print("=" * 80)
print("Fcr-only baseline")
print("=" * 80)
print(f"CSV_PATH: {CSV_PATH}")
print(f"CSV shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

missing_cols = [c for c in VAR_COLS + [FCR_COL] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# Replace inf by nan, then remove bad rows.
df = df.replace([np.inf, -np.inf], np.nan)
n_before = len(df)
df = df.dropna(subset=VAR_COLS + [FCR_COL]).reset_index(drop=True)
n_after = len(df)

if n_after < n_before:
    print(f"Removed {n_before - n_after} rows containing NaN/Inf.")

X = df[VAR_COLS].values.astype(np.float32)
F = df[FCR_COL].values.astype(np.float32).reshape(-1, 1)

check_finite_array("X_raw", X)
check_finite_array("F_raw", F)

print(f"X shape: {X.shape}")
print(f"Fcr shape: {F.shape}")
print(f"Fcr range: min={F.min():.6f}, max={F.max():.6f}, mean={F.mean():.6f}, std={F.std():.6f}")

# Split exactly like proposed model style.
X_tr, X_te, F_tr, F_te = train_test_split(
    X, F,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

X_tr, X_va, F_tr, F_va = train_test_split(
    X_tr, F_tr,
    test_size=VAL_SIZE,
    random_state=SEED,
    shuffle=True,
)

# Normalize X and Fcr exactly.
scX = StandardScaler()
scF = StandardScaler()

X_tr_s = scX.fit_transform(X_tr).astype(np.float32)
X_va_s = scX.transform(X_va).astype(np.float32)
X_te_s = scX.transform(X_te).astype(np.float32)

F_tr_s = scF.fit_transform(F_tr).astype(np.float32)
F_va_s = scF.transform(F_va).astype(np.float32)
F_te_s = scF.transform(F_te).astype(np.float32)

check_finite_array("X_tr_s", X_tr_s)
check_finite_array("X_va_s", X_va_s)
check_finite_array("X_te_s", X_te_s)

check_finite_array("F_tr_s", F_tr_s)
check_finite_array("F_va_s", F_va_s)
check_finite_array("F_te_s", F_te_s)

print(f"Scaled Fcr train: mean={F_tr_s.mean():.6f}, std={F_tr_s.std():.6f}")

dl_tr = make_loader(X_tr_s, F_tr_s, BATCH_SIZE, shuffle=True)
dl_va = make_loader(X_va_s, F_va_s, BATCH_SIZE, shuffle=False)
dl_te = make_loader(X_te_s, F_te_s, BATCH_SIZE, shuffle=False)

model = FcrOnlyMLP(
    in_dim=len(VAR_COLS),
    enc=ENC,
    act=ACT,
    dropout=DROPOUT,
    batchnorm=BATCHNORM,
).to(device)

p_all, p_trainable = count_params(model)

print("=" * 80)
print(f"Model: {len(VAR_COLS)} -> {ENC} -> 1")
print(f"Params: total={p_all:,} | trainable={p_trainable:,} | device={device}")
print(f"n_train={len(X_tr_s):,} | n_val={len(X_va_s):,} | n_test={len(X_te_s):,}")
print("=" * 80)

criterion = nn.MSELoss()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,
    mode="min",
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    min_lr=LR_MIN,
)

# New AMP API, avoids annoying warning.
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(USE_AMP and device.startswith("cuda")),
)

best_val = float("inf")
best_state = None
best_epoch = 0
bad = 0
history = []

sync()
t_train0 = time.perf_counter()

for ep in range(1, EPOCHS + 1):
    sync()
    t0 = time.perf_counter()

    model.train()

    train_sse = 0.0
    train_n = 0

    for Xb, yb in dl_tr:
        Xb = Xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            "cuda",
            enabled=(USE_AMP and device.startswith("cuda")),
        ):
            pred = model(Xb)
            loss = criterion(pred, yb)

        if not torch.isfinite(loss):
            print("[Warning] NaN/Inf loss detected. Turning off AMP is recommended.")
            raise FloatingPointError("Training loss became NaN/Inf.")

        scaler.scale(loss).backward()

        scaler.unscale_(opt)
        if GRAD_CLIP is not None and GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(opt)
        scaler.update()

        train_sse += loss.detach().item() * Xb.size(0)
        train_n += Xb.size(0)

    train_mse = train_sse / max(train_n, 1)
    val_mse = eval_mse_scaled(model, dl_va)

    scheduler.step(val_mse)

    improved = val_mse < best_val - ES_MIN_DELTA

    if improved:
        best_val = val_mse
        best_epoch = ep
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        bad = 0
    else:
        bad += 1

    sync()
    t1 = time.perf_counter()

    row = {
        "epoch": ep,
        "train_mse_scaled": train_mse,
        "train_rmse_scaled": train_mse ** 0.5,
        "val_mse_scaled": val_mse,
        "val_rmse_scaled": val_mse ** 0.5,
        "lr": opt.param_groups[0]["lr"],
        "epoch_time_s": t1 - t0,
        "best_val_mse_scaled": best_val,
        "bad": bad,
    }
    history.append(row)

    print(
        f"Ep {ep:03d} | "
        f"tr_mse={train_mse:.6e} | "
        f"va_mse={val_mse:.6e} | "
        f"va_rmse={val_mse**0.5:.6e} | "
        f"lr={opt.param_groups[0]['lr']:.2e} | "
        f"best={best_val:.6e} | "
        f"bad={bad}/{ES_PATIENCE} | "
        f"time={t1-t0:.2f}s"
    )

    if bad >= ES_PATIENCE:
        print(f"[EarlyStop] epoch={ep} | best_epoch={best_epoch} | best_val_mse={best_val:.6e}")
        break

sync()
t_train1 = time.perf_counter()

training_time = t_train1 - t_train0
epochs_ran = ep

print("=" * 80)
print(f"Train time: {training_time:.3f}s")
print(f"Epochs ran: {epochs_ran}")
print(f"Best epoch: {best_epoch}")
print(f"Best val MSE scaled: {best_val:.6e}")
print("=" * 80)

if best_state is not None:
    model.load_state_dict(best_state)

# ============================================================
# Final metrics in original scale
# ============================================================

y_va_true, y_va_pred = predict_original_scale(model, dl_va, scF)
y_te_true, y_te_pred = predict_original_scale(model, dl_te, scF)

val_metrics = compute_metrics(y_va_true, y_va_pred)
test_metrics = compute_metrics(y_te_true, y_te_pred)

forward_time, n_forward = forward_time_fullset(model, dl_te)

summary = {
    "model_name": "FcrOnlyMLP_parameter_matched",
    "CSV_PATH": CSV_PATH,
    "VAR_COLS": VAR_COLS,
    "FCR_COL": FCR_COL,

    "ENC": ENC,
    "ACT": ACT,
    "DROPOUT": DROPOUT,
    "BATCHNORM": BATCHNORM,

    "params_total": p_all,
    "params_trainable": p_trainable,

    "n_train": len(X_tr_s),
    "n_val": len(X_va_s),
    "n_test": len(X_te_s),

    "EPOCHS": EPOCHS,
    "epochs_ran": epochs_ran,
    "best_epoch": best_epoch,

    "BATCH_SIZE": BATCH_SIZE,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,

    "training_time_seconds": float(training_time),
    "prediction_time_seconds": float(forward_time),
    "prediction_ms_per_sample": float(forward_time / n_forward * 1000.0),
    "prediction_samples_per_second": float(n_forward / forward_time),

    "device": device,
    "gpu_name": torch.cuda.get_device_name(0) if device.startswith("cuda") else None,

    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
}

print("VAL metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v}")

print("TEST metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

print(
    f"TEST forward time: {forward_time:.6f}s | "
    f"{forward_time / n_forward * 1000:.6f} ms/sample | "
    f"{n_forward / forward_time:.2f} samples/s"
)

# ============================================================
# Save outputs
# ============================================================

history_df = pd.DataFrame(history)
history_df.to_csv(f"{OUT_DIR}/tables/training_history.csv", index=False)

metrics_flat = {
    "training_time_seconds": float(training_time),
    "prediction_time_seconds": float(forward_time),
    "prediction_ms_per_sample": float(forward_time / n_forward * 1000.0),
    "prediction_samples_per_second": float(n_forward / forward_time),
    "params_total": int(p_all),
    "params_trainable": int(p_trainable),
    "best_epoch": int(best_epoch),
    "epochs_ran": int(epochs_ran),
}

for k, v in val_metrics.items():
    metrics_flat[f"VAL_{k}"] = v

for k, v in test_metrics.items():
    metrics_flat[f"TEST_{k}"] = v

pd.DataFrame.from_dict(metrics_flat, orient="index", columns=["value"]).to_csv(
    f"{OUT_DIR}/tables/metrics_summary.csv"
)

with open(f"{OUT_DIR}/tables/metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

pred_df = pd.DataFrame({
    "Fcr_true": y_te_true,
    "Fcr_pred": y_te_pred,
    "abs_error": np.abs(y_te_pred - y_te_true),
    "rel_error": np.abs(y_te_pred - y_te_true) / np.maximum(np.abs(y_te_true), 1e-12),
})
pred_df.to_csv(f"{OUT_DIR}/tables/test_predictions_Fcr.csv", index=False)

torch.save(
    {
        "model_name": "FcrOnlyMLP_parameter_matched",
        "model_state_dict": model.state_dict(),
        "config": {
            "VAR_COLS": VAR_COLS,
            "FCR_COL": FCR_COL,
            "ENC": ENC,
            "ACT": ACT,
            "DROPOUT": DROPOUT,
            "BATCHNORM": BATCHNORM,
            "params_total": p_all,
            "params_trainable": p_trainable,
        },
        "summary": summary,
    },
    f"{OUT_DIR}/checkpoints/best_model_full_checkpoint.pt",
)

torch.save(
    model.state_dict(),
    f"{OUT_DIR}/checkpoints/best_model_state_dict.pt",
)

joblib.dump(scX, f"{OUT_DIR}/scalers/scX.joblib")
joblib.dump(scF, f"{OUT_DIR}/scalers/scF.joblib")

print("=" * 80)
print("Saved outputs:")
print(f"- {OUT_DIR}/checkpoints/best_model_full_checkpoint.pt")
print(f"- {OUT_DIR}/checkpoints/best_model_state_dict.pt")
print(f"- {OUT_DIR}/scalers/scX.joblib")
print(f"- {OUT_DIR}/scalers/scF.joblib")
print(f"- {OUT_DIR}/tables/training_history.csv")
print(f"- {OUT_DIR}/tables/metrics_summary.csv")
print(f"- {OUT_DIR}/tables/metrics_summary.json")
print(f"- {OUT_DIR}/tables/test_predictions_Fcr.csv")
print("=" * 80)

Fcr-only baseline
CSV_PATH: DatasetMTH_coeffs_DT123_5levels.csv
CSV shape: (2343750, 14)
Columns: ['var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'var7', 'var8', 'var9', 'var10', 'Fcr', 'Lambda0', 'Lambda1', 'Lambda2']
X shape: (2343750, 10)
Fcr shape: (2343750, 1)
Fcr range: min=1.174934, max=10.766446, mean=2.880636, std=1.200269
Scaled Fcr train: mean=-0.000000, std=1.000000
Model: 10 -> [238, 263, 263] -> 1
Params: total=135,171 | trainable=135,171 | device=cuda
n_train=1,687,500 | n_val=187,500 | n_test=468,750
Ep 001 | tr_mse=3.012259e-02 | va_mse=1.143479e-03 | va_rmse=3.381537e-02 | lr=1.00e-03 | best=1.143479e-03 | bad=0/8 | time=19.15s
Ep 002 | tr_mse=1.025842e-03 | va_mse=7.884038e-04 | va_rmse=2.807853e-02 | lr=1.00e-03 | best=7.884038e-04 | bad=0/8 | time=17.61s
Ep 003 | tr_mse=6.294457e-04 | va_mse=4.116639e-04 | va_rmse=2.028950e-02 | lr=1.00e-03 | best=4.116639e-04 | bad=0/8 | time=18.03s
Ep 004 | tr_mse=4.794573e-04 | va_mse=4.324122e-04 | va_rmse=2.079452e-02 | lr=1.

In [ ]:
CSV_PATH = "/content/drive/MyDrive/NN Multihead 2025/DatasetMTH_coeffs.csv"

In [ ]:
# ============================================================
# Baseline NN-only postbuckling regressor
# Input : var1..var10
# Output: [Lambda0, Lambda1, Lambda2]
#
# Parameter-matched with proposed PCNN:
#   10 -> 256 -> 256 -> 256 -> 3
#   total params = 135,171
#
# Normalization exactly like proposed model:
#   X -> scX
#   Y -> scY
# ============================================================

import os
import json
import time
import random
import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# ============================================================
# Config
# ============================================================

CSV_PATH = "DatasetMTH_coeffs_DT123_5levels.csv"

OUT_DIR = "baseline_postbuckling_only_param_matched"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)
os.makedirs(f"{OUT_DIR}/scalers", exist_ok=True)
os.makedirs(f"{OUT_DIR}/arrays", exist_ok=True)

VAR_COLS = [f"var{i}" for i in range(1, 11)]

# New dataset column names
Y_COLS = ["Lambda0", "Lambda1", "Lambda2"]

# For curve reconstruction
Nw = 50
W_GRID = np.linspace(0.0, 1.0, Nw).astype(np.float32)

# Exact parameter-matched postbuckling-only MLP:
# 10 -> 256 -> 256 -> 256 -> 3 = 135,171 params
ENC = [256, 256, 256]

ACT = "relu"
DROPOUT = 0.0
BATCHNORM = False

EPOCHS = 200
BATCH_SIZE = 4096
LR = 1e-3
WEIGHT_DECAY = 1e-4

TEST_SIZE = 0.2
VAL_SIZE = 0.1
SEED = 42

ES_PATIENCE = 8
ES_MIN_DELTA = 0.5e-5

LR_FACTOR = 0.5
LR_PATIENCE = 4
LR_MIN = 1e-6

USE_AMP = True
GRAD_CLIP = 1.0

device = "cuda" if torch.cuda.is_available() else "cpu"
pin_memory = device.startswith("cuda")


# ============================================================
# Utils
# ============================================================

def seed_everything(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True


def sync():
    if device.startswith("cuda"):
        torch.cuda.synchronize()


def get_act(name):
    name = name.lower()

    if name == "relu":
        return nn.ReLU(inplace=True)
    if name == "gelu":
        return nn.GELU()
    if name == "silu":
        return nn.SiLU(inplace=True)
    if name == "tanh":
        return nn.Tanh()
    if name == "leaky_relu":
        return nn.LeakyReLU(0.1, inplace=True)
    if name == "elu":
        return nn.ELU(inplace=True)
    if name == "softplus":
        return nn.Softplus()

    raise ValueError(f"Unknown activation: {name}")


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def check_finite_array(name, arr):
    arr = np.asarray(arr)
    if not np.isfinite(arr).all():
        n_nan = np.isnan(arr).sum()
        n_inf = np.isinf(arr).sum()
        raise ValueError(f"{name} contains NaN/Inf: NaN={n_nan}, Inf={n_inf}")


def make_loader(X, y, batch_size, shuffle):
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)

    ds = TensorDataset(X, y)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        pin_memory=pin_memory,
        num_workers=0,
    )


# ============================================================
# Model
# ============================================================

class PostbucklingOnlyMLP(nn.Module):
    def __init__(self, in_dim, out_dim, enc, act="gelu", dropout=0.0, batchnorm=False):
        super().__init__()

        layers = []
        prev = in_dim

        for h in enc:
            layers.append(nn.Linear(prev, h))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(get_act(act))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def eval_mse_scaled(model, loader):
    """
    MSE averaged over all samples and all output dimensions.
    This matches nn.MSELoss behavior.
    """
    model.eval()

    sse = 0.0
    n_total = 0

    for Xb, yb in loader:
        Xb = Xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        pred = model(Xb)

        if not torch.isfinite(pred).all():
            raise FloatingPointError("Prediction contains NaN/Inf during evaluation.")

        diff2 = (pred - yb) ** 2
        sse += diff2.sum().item()
        n_total += diff2.numel()

    return sse / max(n_total, 1)


@torch.no_grad()
def predict_original_scale(model, loader, scY):
    model.eval()

    y_true_s_all = []
    y_pred_s_all = []

    for Xb, yb in loader:
        Xb = Xb.to(device, non_blocking=True)

        pred_s = model(Xb).detach().cpu().numpy()

        y_pred_s_all.append(pred_s)
        y_true_s_all.append(yb.numpy())

    y_true_s = np.vstack(y_true_s_all)
    y_pred_s = np.vstack(y_pred_s_all)

    y_true = scY.inverse_transform(y_true_s).astype(np.float64)
    y_pred = scY.inverse_transform(y_pred_s).astype(np.float64)

    return y_true, y_pred


def compute_coeff_metrics(y_true, y_pred, names=("Lambda0", "Lambda1", "Lambda2")):
    metrics = {}

    for j, name in enumerate(names):
        yt = y_true[:, j]
        yp = y_pred[:, j]

        abs_err = np.abs(yp - yt)
        rel_err = abs_err / np.maximum(np.abs(yt), 1e-12)

        metrics[f"{name}_MAE"] = float(mean_absolute_error(yt, yp))
        metrics[f"{name}_RMSE"] = float(mean_squared_error(yt, yp) ** 0.5)
        metrics[f"{name}_R2"] = float(r2_score(yt, yp))

        metrics[f"{name}_mean_abs_error"] = float(np.mean(abs_err))
        metrics[f"{name}_median_abs_error"] = float(np.median(abs_err))
        metrics[f"{name}_p95_abs_error"] = float(np.percentile(abs_err, 95))
        metrics[f"{name}_max_abs_error"] = float(np.max(abs_err))

        metrics[f"{name}_mean_rel_error"] = float(np.mean(rel_err))
        metrics[f"{name}_median_rel_error"] = float(np.median(rel_err))
        metrics[f"{name}_p95_rel_error"] = float(np.percentile(rel_err, 95))
        metrics[f"{name}_max_rel_error"] = float(np.max(rel_err))

    return metrics


def reconstruct_curve(Y, W_GRID):
    """
    Y shape: (N,3), columns [Lambda0, Lambda1, Lambda2]
    curve = Lambda0 + Lambda1*W + Lambda2*W^2
    """
    W = W_GRID.reshape(1, -1).astype(np.float64)
    Y = Y.astype(np.float64)

    curve = (
        Y[:, 0:1]
        + Y[:, 1:2] * W
        + Y[:, 2:3] * (W ** 2)
    )

    return curve


def compute_curve_metrics(y_true, y_pred, W_GRID):
    curve_true = reconstruct_curve(y_true, W_GRID)
    curve_pred = reconstruct_curve(y_pred, W_GRID)

    diff = curve_pred - curve_true
    abs_diff = np.abs(diff)

    rmse_each = np.sqrt(np.mean(diff ** 2, axis=1))
    mae_each = np.mean(abs_diff, axis=1)

    rel_l2_each = (
        np.linalg.norm(diff, axis=1)
        / np.maximum(np.linalg.norm(curve_true, axis=1), 1e-12)
    )

    max_abs_each = np.max(abs_diff, axis=1)

    metrics = {
        "Curve_RMSE_mean": float(np.mean(rmse_each)),
        "Curve_RMSE_median": float(np.median(rmse_each)),
        "Curve_RMSE_p95": float(np.percentile(rmse_each, 95)),
        "Curve_RMSE_max": float(np.max(rmse_each)),

        "Curve_MAE_mean": float(np.mean(mae_each)),
        "Curve_MAE_median": float(np.median(mae_each)),
        "Curve_MAE_p95": float(np.percentile(mae_each, 95)),
        "Curve_MAE_max": float(np.max(mae_each)),

        "Curve_rel_L2_mean": float(np.mean(rel_l2_each)),
        "Curve_rel_L2_median": float(np.median(rel_l2_each)),
        "Curve_rel_L2_p95": float(np.percentile(rel_l2_each, 95)),
        "Curve_rel_L2_max": float(np.max(rel_l2_each)),

        "Curve_max_abs_error_mean": float(np.mean(max_abs_each)),
        "Curve_max_abs_error_median": float(np.median(max_abs_each)),
        "Curve_max_abs_error_p95": float(np.percentile(max_abs_each, 95)),
        "Curve_max_abs_error_max": float(np.max(max_abs_each)),
    }

    return metrics, curve_true, curve_pred


@torch.no_grad()
def forward_time_fullset(model, loader, n_warmup_batches=5):
    model.eval()

    # warm-up
    it = iter(loader)
    for _ in range(n_warmup_batches):
        try:
            Xb, _ = next(it)
        except StopIteration:
            break

        Xb = Xb.to(device, non_blocking=True)
        _ = model(Xb)

    sync()

    t0 = time.perf_counter()

    n = 0
    for Xb, _ in loader:
        Xb = Xb.to(device, non_blocking=True)
        _ = model(Xb)
        n += Xb.size(0)

    sync()

    t1 = time.perf_counter()

    return t1 - t0, n


# ============================================================
# Main
# ============================================================

seed_everything(SEED, deterministic=False)

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

print("=" * 80)
print("Postbuckling-only baseline")
print("=" * 80)
print(f"CSV_PATH: {CSV_PATH}")
print(f"CSV shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

missing_cols = [c for c in VAR_COLS + Y_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

df = df.replace([np.inf, -np.inf], np.nan)

n_before = len(df)
df = df.dropna(subset=VAR_COLS + Y_COLS).reset_index(drop=True)
n_after = len(df)

if n_after < n_before:
    print(f"Removed {n_before - n_after} rows containing NaN/Inf.")

X = df[VAR_COLS].values.astype(np.float32)
Y = df[Y_COLS].values.astype(np.float32)

check_finite_array("X_raw", X)
check_finite_array("Y_raw", Y)

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"Y columns: {Y_COLS}")

for j, name in enumerate(Y_COLS):
    print(
        f"{name}: min={Y[:,j].min():.6f}, "
        f"max={Y[:,j].max():.6f}, "
        f"mean={Y[:,j].mean():.6f}, "
        f"std={Y[:,j].std():.6f}"
    )

# Split
X_tr, X_te, Y_tr, Y_te = train_test_split(
    X, Y,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

X_tr, X_va, Y_tr, Y_va = train_test_split(
    X_tr, Y_tr,
    test_size=VAL_SIZE,
    random_state=SEED,
    shuffle=True,
)

# Normalize X and Y using training set only
scX = StandardScaler()
scY = StandardScaler()

X_tr_s = scX.fit_transform(X_tr).astype(np.float32)
X_va_s = scX.transform(X_va).astype(np.float32)
X_te_s = scX.transform(X_te).astype(np.float32)

Y_tr_s = scY.fit_transform(Y_tr).astype(np.float32)
Y_va_s = scY.transform(Y_va).astype(np.float32)
Y_te_s = scY.transform(Y_te).astype(np.float32)

check_finite_array("X_tr_s", X_tr_s)
check_finite_array("X_va_s", X_va_s)
check_finite_array("X_te_s", X_te_s)

check_finite_array("Y_tr_s", Y_tr_s)
check_finite_array("Y_va_s", Y_va_s)
check_finite_array("Y_te_s", Y_te_s)

print("Scaled Y train stats:")
for j, name in enumerate(Y_COLS):
    print(f"{name}_scaled: mean={Y_tr_s[:,j].mean():.6f}, std={Y_tr_s[:,j].std():.6f}")

dl_tr = make_loader(X_tr_s, Y_tr_s, BATCH_SIZE, shuffle=True)
dl_va = make_loader(X_va_s, Y_va_s, BATCH_SIZE, shuffle=False)
dl_te = make_loader(X_te_s, Y_te_s, BATCH_SIZE, shuffle=False)

model = PostbucklingOnlyMLP(
    in_dim=len(VAR_COLS),
    out_dim=len(Y_COLS),
    enc=ENC,
    act=ACT,
    dropout=DROPOUT,
    batchnorm=BATCHNORM,
).to(device)

p_all, p_trainable = count_params(model)

print("=" * 80)
print(f"Model: {len(VAR_COLS)} -> {ENC} -> {len(Y_COLS)}")
print(f"Params: total={p_all:,} | trainable={p_trainable:,} | device={device}")
print(f"n_train={len(X_tr_s):,} | n_val={len(X_va_s):,} | n_test={len(X_te_s):,}")
print("=" * 80)

criterion = nn.MSELoss()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,
    mode="min",
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    min_lr=LR_MIN,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(USE_AMP and device.startswith("cuda")),
)

best_val = float("inf")
best_state = None
best_epoch = 0
bad = 0
history = []

sync()
t_train0 = time.perf_counter()

for ep in range(1, EPOCHS + 1):
    sync()
    t0 = time.perf_counter()

    model.train()

    train_sse = 0.0
    train_numel = 0

    for Xb, yb in dl_tr:
        Xb = Xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            "cuda",
            enabled=(USE_AMP and device.startswith("cuda")),
        ):
            pred = model(Xb)
            loss = criterion(pred, yb)

        if not torch.isfinite(loss):
            print("[Warning] NaN/Inf loss detected.")
            raise FloatingPointError("Training loss became NaN/Inf.")

        scaler.scale(loss).backward()

        scaler.unscale_(opt)
        if GRAD_CLIP is not None and GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(opt)
        scaler.update()

        train_sse += torch.sum((pred.detach() - yb.detach()) ** 2).item()
        train_numel += pred.numel()

    train_mse = train_sse / max(train_numel, 1)
    val_mse = eval_mse_scaled(model, dl_va)

    scheduler.step(val_mse)

    improved = val_mse < best_val - ES_MIN_DELTA

    if improved:
        best_val = val_mse
        best_epoch = ep
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        bad = 0
    else:
        bad += 1

    sync()
    t1 = time.perf_counter()

    row = {
        "epoch": ep,
        "train_mse_scaled": train_mse,
        "train_rmse_scaled": train_mse ** 0.5,
        "val_mse_scaled": val_mse,
        "val_rmse_scaled": val_mse ** 0.5,
        "lr": opt.param_groups[0]["lr"],
        "epoch_time_s": t1 - t0,
        "best_val_mse_scaled": best_val,
        "bad": bad,
    }
    history.append(row)

    print(
        f"Ep {ep:03d} | "
        f"tr_mse={train_mse:.6e} | "
        f"va_mse={val_mse:.6e} | "
        f"va_rmse={val_mse**0.5:.6e} | "
        f"lr={opt.param_groups[0]['lr']:.2e} | "
        f"best={best_val:.6e} | "
        f"bad={bad}/{ES_PATIENCE} | "
        f"time={t1-t0:.2f}s"
    )

    if bad >= ES_PATIENCE:
        print(f"[EarlyStop] epoch={ep} | best_epoch={best_epoch} | best_val_mse={best_val:.6e}")
        break

sync()
t_train1 = time.perf_counter()

training_time = t_train1 - t_train0
epochs_ran = ep

print("=" * 80)
print(f"Train time: {training_time:.3f}s")
print(f"Epochs ran: {epochs_ran}")
print(f"Best epoch: {best_epoch}")
print(f"Best val MSE scaled: {best_val:.6e}")
print("=" * 80)

if best_state is not None:
    model.load_state_dict(best_state)


# ============================================================
# Final metrics in original scale
# ============================================================

Y_va_true, Y_va_pred = predict_original_scale(model, dl_va, scY)
Y_te_true, Y_te_pred = predict_original_scale(model, dl_te, scY)

val_coeff_metrics = compute_coeff_metrics(Y_va_true, Y_va_pred, names=Y_COLS)
test_coeff_metrics = compute_coeff_metrics(Y_te_true, Y_te_pred, names=Y_COLS)

val_curve_metrics, curve_va_true, curve_va_pred = compute_curve_metrics(
    Y_va_true, Y_va_pred, W_GRID
)

test_curve_metrics, curve_te_true, curve_te_pred = compute_curve_metrics(
    Y_te_true, Y_te_pred, W_GRID
)

forward_time, n_forward = forward_time_fullset(model, dl_te)

summary = {
    "model_name": "PostbucklingOnlyMLP_parameter_matched",
    "CSV_PATH": CSV_PATH,
    "VAR_COLS": VAR_COLS,
    "Y_COLS": Y_COLS,

    "ENC": ENC,
    "ACT": ACT,
    "DROPOUT": DROPOUT,
    "BATCHNORM": BATCHNORM,

    "params_total": p_all,
    "params_trainable": p_trainable,

    "n_train": len(X_tr_s),
    "n_val": len(X_va_s),
    "n_test": len(X_te_s),

    "EPOCHS": EPOCHS,
    "epochs_ran": epochs_ran,
    "best_epoch": best_epoch,

    "BATCH_SIZE": BATCH_SIZE,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,

    "training_time_seconds": float(training_time),
    "prediction_time_seconds": float(forward_time),
    "prediction_ms_per_sample": float(forward_time / n_forward * 1000.0),
    "prediction_samples_per_second": float(n_forward / forward_time),

    "device": device,
    "gpu_name": torch.cuda.get_device_name(0) if device.startswith("cuda") else None,

    "val_coeff_metrics": val_coeff_metrics,
    "val_curve_metrics": val_curve_metrics,

    "test_coeff_metrics": test_coeff_metrics,
    "test_curve_metrics": test_curve_metrics,
}

print("VAL coefficient metrics:")
for k, v in val_coeff_metrics.items():
    print(f"{k}: {v}")

print("VAL curve metrics:")
for k, v in val_curve_metrics.items():
    print(f"{k}: {v}")

print("TEST coefficient metrics:")
for k, v in test_coeff_metrics.items():
    print(f"{k}: {v}")

print("TEST curve metrics:")
for k, v in test_curve_metrics.items():
    print(f"{k}: {v}")

print(
    f"TEST forward time: {forward_time:.6f}s | "
    f"{forward_time / n_forward * 1000:.6f} ms/sample | "
    f"{n_forward / forward_time:.2f} samples/s"
)


# ============================================================
# Save outputs
# ============================================================

history_df = pd.DataFrame(history)
history_df.to_csv(f"{OUT_DIR}/tables/training_history.csv", index=False)

metrics_flat = {
    "training_time_seconds": float(training_time),
    "prediction_time_seconds": float(forward_time),
    "prediction_ms_per_sample": float(forward_time / n_forward * 1000.0),
    "prediction_samples_per_second": float(n_forward / forward_time),
    "params_total": int(p_all),
    "params_trainable": int(p_trainable),
    "best_epoch": int(best_epoch),
    "epochs_ran": int(epochs_ran),
}

for k, v in val_coeff_metrics.items():
    metrics_flat[f"VAL_{k}"] = v

for k, v in val_curve_metrics.items():
    metrics_flat[f"VAL_{k}"] = v

for k, v in test_coeff_metrics.items():
    metrics_flat[f"TEST_{k}"] = v

for k, v in test_curve_metrics.items():
    metrics_flat[f"TEST_{k}"] = v

pd.DataFrame.from_dict(metrics_flat, orient="index", columns=["value"]).to_csv(
    f"{OUT_DIR}/tables/metrics_summary.csv"
)

with open(f"{OUT_DIR}/tables/metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

pred_df = pd.DataFrame({
    "Lambda0_true": Y_te_true[:, 0],
    "Lambda0_pred": Y_te_pred[:, 0],
    "Lambda1_true": Y_te_true[:, 1],
    "Lambda1_pred": Y_te_pred[:, 1],
    "Lambda2_true": Y_te_true[:, 2],
    "Lambda2_pred": Y_te_pred[:, 2],
})

pred_df["Lambda0_abs_error"] = np.abs(pred_df["Lambda0_pred"] - pred_df["Lambda0_true"])
pred_df["Lambda1_abs_error"] = np.abs(pred_df["Lambda1_pred"] - pred_df["Lambda1_true"])
pred_df["Lambda2_abs_error"] = np.abs(pred_df["Lambda2_pred"] - pred_df["Lambda2_true"])

pred_df.to_csv(f"{OUT_DIR}/tables/test_predictions_coefficients.csv", index=False)

curve_npz_path = f"{OUT_DIR}/arrays/test_curves_true_pred.npz"
np.savez_compressed(
    curve_npz_path,
    W_GRID=W_GRID,
    curve_true=curve_te_true.astype(np.float32),
    curve_pred=curve_te_pred.astype(np.float32),
    Y_true=Y_te_true.astype(np.float32),
    Y_pred=Y_te_pred.astype(np.float32),
)

torch.save(
    {
        "model_name": "PostbucklingOnlyMLP_parameter_matched",
        "model_state_dict": model.state_dict(),
        "config": {
            "VAR_COLS": VAR_COLS,
            "Y_COLS": Y_COLS,
            "ENC": ENC,
            "ACT": ACT,
            "DROPOUT": DROPOUT,
            "BATCHNORM": BATCHNORM,
            "params_total": p_all,
            "params_trainable": p_trainable,
            "W_GRID": W_GRID.tolist(),
        },
        "summary": summary,
    },
    f"{OUT_DIR}/checkpoints/best_model_full_checkpoint.pt",
)

torch.save(
    model.state_dict(),
    f"{OUT_DIR}/checkpoints/best_model_state_dict.pt",
)

joblib.dump(scX, f"{OUT_DIR}/scalers/scX.joblib")
joblib.dump(scY, f"{OUT_DIR}/scalers/scY.joblib")

print("=" * 80)
print("Saved outputs:")
print(f"- {OUT_DIR}/checkpoints/best_model_full_checkpoint.pt")
print(f"- {OUT_DIR}/checkpoints/best_model_state_dict.pt")
print(f"- {OUT_DIR}/scalers/scX.joblib")
print(f"- {OUT_DIR}/scalers/scY.joblib")
print(f"- {OUT_DIR}/tables/training_history.csv")
print(f"- {OUT_DIR}/tables/metrics_summary.csv")
print(f"- {OUT_DIR}/tables/metrics_summary.json")
print(f"- {OUT_DIR}/tables/test_predictions_coefficients.csv")
print(f"- {curve_npz_path}")
print("=" * 80)

Postbuckling-only baseline
CSV_PATH: DatasetMTH_coeffs_DT123_5levels.csv
CSV shape: (2343750, 14)
Columns: ['var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'var7', 'var8', 'var9', 'var10', 'Fcr', 'Lambda0', 'Lambda1', 'Lambda2']
X shape: (2343750, 10)
Y shape: (2343750, 3)
Y columns: ['Lambda0', 'Lambda1', 'Lambda2']
Lambda0: min=1.174934, max=10.766446, mean=2.880636, std=1.200269
Lambda1: min=-0.000000, max=0.000000, mean=0.000000, std=0.000000
Lambda2: min=0.510530, max=2.084311, mean=0.836040, std=0.284696
Scaled Y train stats:
Lambda0_scaled: mean=-0.000000, std=1.000000
Lambda1_scaled: mean=-0.000000, std=1.000000
Lambda2_scaled: mean=-0.000000, std=1.000000
Model: 10 -> [256, 256, 256] -> 3
Params: total=135,171 | trainable=135,171 | device=cuda
n_train=1,687,500 | n_val=187,500 | n_test=468,750
Ep 001 | tr_mse=2.641754e-01 | va_mse=1.479679e-01 | va_rmse=3.846659e-01 | lr=1.00e-03 | best=1.479679e-01 | bad=0/8 | time=17.29s
Ep 002 | tr_mse=7.994376e-02 | va_mse=3.565837e-02 | v